# Task 2B - 03 Priorities and bottlenecks
What does each order cost the business if it is not served today, and which resource actually limits service?

In [1]:
import sys, warnings; warnings.filterwarnings('ignore')
sys.path.append('../../tools/task2b')
import numpy as np, pandas as pd, matplotlib.pyplot as plt, json, time
from task2b_common import *
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 50); pd.set_option('display.max_rows', 120)
plt.rcParams.update({'figure.dpi': 90, 'axes.grid': True, 'grid.alpha': .25})
for d in (METRICS, FIGURES, INTERIM): d.mkdir(parents=True, exist_ok=True)
scn, fleet, veh, dt, al = load_tables()
sub_team = pd.read_csv(SUBMISSION)            # the teammate's submitted allocation (read-only)
import task2b_exact as X
S = X.S


## 1. Two ways to express priority
**(a) The weighted score used by the first solver** (teammate's `solve_task2b.py`):
`value = 10 per stop + 1 x m3 + 5 if chilled + 20 if deferred yesterday + 4 x (days since last served - 1)`.

**(b) The written lexicographic policy** (the team's stated order): serve first the Fresh orders **deferred yesterday**, then **chilled** orders, then those with the highest **days since last served**; volume only as a tie-break. Each level is optimised first and then *fixed* before the next one is considered, so a lower level can never overrule a higher one.

The two coincide when no real trade-off exists and differ exactly where the day is tight; notebook 05 quantifies that.

In [2]:
orders = pd.DataFrame([dict(order_ref=o.ref, outlet=o.outlet, brand=o.brand, district=o.district, chilled=o.chilled, m3=o.m3,
                              deferred_yesterday=o.deferred_yday, days_since=o.days_since, value=S.order_value(o)) for o in S.ORDERS.values()])
print('weights:', dict(base=S.W_BASE, volume=S.W_VOLUME, chilled=S.W_CHILLED, deferred_yday=S.W_DEFERRED_YDAY, days_since=S.W_DAYS_SINCE))
print('\nhighest-priority orders:'); print(orders.sort_values('value', ascending=False).head(8).round(1).to_string(index=False))
print('\norders skipped yesterday:'); print(orders[orders.deferred_yesterday == 1].round(1).to_string(index=False))
print('\ndays since last served:', orders.days_since.value_counts().sort_index().to_dict())

weights: {'base': 10.0, 'volume': 1.0, 'chilled': 5.0, 'deferred_yday': 20.0, 'days_since': 4.0}

highest-priority orders:
order_ref outlet brand   district  chilled   m3  deferred_yesterday  days_since  value
   S1-083 OUT074 Fresh   Puttalam     True  8.7                   1           5   59.7
   S1-079 OUT071 Style Kurunegala    False  9.4                   1           5   55.4
   S1-078 OUT070 Style Kurunegala    False 40.7                   0           2   54.7
   S1-068 OUT063 Style     Matara    False  7.5                   1           5   53.5
   S1-038 OUT032 Fresh    Gampaha     True 14.4                   1           2   53.4
   S1-023 OUT022  Tech    Colombo    False  5.4                   1           5   51.4
   S1-025 OUT024  Tech    Colombo    False  4.7                   1           5   50.7
   S1-041 OUT034 Fresh    Gampaha     True  6.7                   1           2   45.7

orders skipped yesterday:
order_ref outlet brand   district  chilled   m3  deferred_yesterday

**Finding.** Ten orders were already skipped yesterday (6 Fresh, 2 Style, 2 Tech), and five orders have gone **5 days** without a delivery. The highest priority of all is S1-083 (OUT074 Puttalam, chilled, skipped yesterday and 5 days unserved). Puttalam is also the farthest district (173 minutes outbound), which sets up the central tension of the day.

## 2. The binding resource: reefer capacity, in volume, weight and time

In [3]:
reefers = [v for v in S.AVAILABLE['S1'] if v.temp == 'reefer']
chilled = [o for o in S.ORDERS.values() if o.chilled]
print('available reefers:', [(v.vid, v.type, v.m3_cap, v.kg_cap) for v in reefers])
vol_dem, kg_dem = sum(o.m3 for o in chilled), sum(o.kg for o in chilled)
vol_cap, kg_cap = sum(v.m3_cap for v in reefers), sum(v.kg_cap for v in reefers)
print('\nchilled demand: %.1f m3, %.1f t (%d orders)' % (vol_dem, kg_dem / 1000, len(chilled)))
print('reefer capacity: %.1f m3 and %.1f t in ONE wave; %.1f m3 with the maximum two trips per vehicle' % (vol_cap, kg_cap / 1000, 2 * vol_cap))
print('=> even two perfect full loads per reefer carry %.0f%% of chilled volume' % (100 * 2 * vol_cap / vol_dem))

available reefers: [('VEH003', 'truck', 26.4, 5510.0), ('VEH006', 'truck', 33.4, 6840.0), ('VEH007', 'truck', 19.4, 3610.0), ('VEH036', 'van', 7.0, 1040.0)]

chilled demand: 181.6 m3, 32.8 t (26 orders)
reefer capacity: 86.2 m3 and 17.0 t in ONE wave; 172.4 m3 with the maximum two trips per vehicle
=> even two perfect full loads per reefer carry 95% of chilled volume


In [4]:
# Minimum reefer minutes needed to serve EVERY chilled order (best possible partition into trips per district, ignoring who drives)
big_reefer = max(reefers, key=lambda v: v.m3_cap); van_reefer = [v for v in reefers if v.type == 'van'][0]
rows = []
for dist in sorted({o.district for o in chilled}):
    g = [o for o in chilled if o.district == dist]
    parts = X._best_partition(g, big_reefer, van_reefer, 'predawn')
    rows.append((dist, len(g), round(sum(o.m3 for o in g), 1), len(parts), sum(i[2] for _, i in parts)))
mt = pd.DataFrame(rows, columns=['district', 'chilled_orders', 'chilled_m3', 'min_trips', 'min_minutes'])
print(mt.to_string(index=False))
need_min, have_min = mt.min_minutes.sum(), len(reefers) * S.BUDGET['predawn']
print('\nminutes needed to serve all chilled orders (a lower bound): %d | pre-dawn minutes the 4 reefers have: %d  -> %.0f%% of what is needed' % (need_min, have_min, 100 * have_min / need_min))
print('trips needed (lower bound):', int(mt.min_trips.sum()), '| trips available (4 reefers x 2):', 2 * len(reefers))

  district  chilled_orders  chilled_m3  min_trips  min_minutes
   Colombo               9        51.6          4        276.0
     Galle               2        20.3          1        143.0
   Gampaha               6        48.1          2        201.0
  Kalutara               3        16.0          1        134.0
Kurunegala               3        25.0          1        210.0
    Matara               2        12.0          1        177.0
  Puttalam               1         8.7          1        188.0

minutes needed to serve all chilled orders (a lower bound): 1329 | pre-dawn minutes the 4 reefers have: 1080  -> 81% of what is needed
trips needed (lower bound): 11 | trips available (4 reefers x 2): 8


**Finding - two independent bottlenecks, each sufficient on its own:**
- **Volume:** chilled demand exceeds even two full loads of every available reefer.
- **Time:** serving every chilled order needs more reefer minutes (and more trips) than the four reefers have in the 270-minute pre-dawn window.

Ambient demand is nowhere near binding (audit, notebook 01). Hence the policy question is *which chilled orders* wait, and there is a hard floor on how many must.